# testing s1

Starts a random training board: `n_gold` uniform in 0–3, opening `any`.

The **Checkpoint** dropdown is every `*.pt` under `weights/`, newest first (the same mtime order as the adapter dropdowns in the other notebooks). **Load** reads that state dict into S1. **Refresh** rescans the folder. **New board** deals another training board (`n_gold` uniform in 0–3, opening `any`) and clears the previous click.

The teal card is the play notebook's game-settings editor. **Edit game settings** opens the form (pose, direction, radii, game size, gold, openings, walls). **Render game settings** writes that form onto the live board. A click is ignored while the form is open.

A click inside the agent disc is passed as that world coordinate. Training labels that point `noop`.

Click the picture. The click is stored as a world coordinate (`clicks`) and a red dot is drawn on the picture. S1 reads the engine frame. The dot is only in the widget.

S1 then emits primitives until it has produced 20 `noop`s in a row. Each printed line is also appended to `moves`:

- the move S1 took
- the four logits, in order `noop`, `CLOCK`, `ANTICLOCK`, `FORWARD`
- the oracle's move for that same point, from the pose before the move
- **think** — the forward pass, including the copy onto the device
- **render** — the new frame after the move (the frame S1 reads on the next step)
- **notebook** — drawing that frame, with the red dot, into the widget

After 20 no-ops the loop stops and another click works. A pursuit also stops after a half turn plus a crossing of the board diagonal, plus 20% (64 moves). The move log scrolls inside its own box; the picture stays put.

Run this on the GPU box. The cell needs the repo environment (`pygame`, `torch`, `ipywidgets`).


In [ ]:
import io
import math
import os
import sys
import time
import traceback

os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")

# Run from the repo root so `agent` / `neural_net` imports resolve.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import ipywidgets as widgets
import numpy as np
import torch
from IPython.display import HTML, display
from PIL import Image

from agent.game_io import (
    apply_edited_settings_dict,
    game_to_settings_dict,
    new_multi_gold_game,
)
from agent.notebook_ui import settings_editor
from neural_net.oracle import ACTIONS, point_oracle
from neural_net.paths import weights_root
from neural_net.render import WallCache, apply_primitive
from neural_net.s1 import S1

# Twenty model no-ops in a row hands the board back.
# A half turn is pi/(pi/30) = 30 primitives. One FORWARD is 1/16 of
# the board, so the unit-square diagonal is ceil(16*sqrt(2)) forwards.
# Stop at 20% past that sum and hand the click back.
NOOP_STOP = 20
_HALF_TURN = 30
_DIAGONAL_FORWARDS = math.ceil(16 * math.sqrt(2))
MAX_MOVES = math.ceil(1.2 * (_HALF_TURN + _DIAGONAL_FORWARDS))
BOARD_PX = 560
DOT_RADIUS = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Every pursuit appends here. `clicks` is the press; `moves` is one row
# per primitive (same numbers as the printed log).
clicks = []
moves = []

state = {
    "busy": False,
    "ignore_through": 0,
    "model": None,
    "game": None,
    "cache": None,
    "frame": None,
    "dot": None,
}


def _checkpoints():
    """Every ``*.pt`` under ``weights/``, newest first.

    Same mtime order as the adapter dropdowns in the other notebooks.
    S1 checkpoints are state dicts (``weights/s1/<run>/step_XXXXXX.pt``
    and ``last.pt``), so the scan is files, not adapter folders.
    """
    root = weights_root()
    if not root.is_dir():
        return []
    files = [path for path in root.rglob("*.pt") if path.is_file()]
    files.sort(key=lambda path: (path.stat().st_mtime, path.as_posix()), reverse=True)
    return files


def _ckpt_options():
    root = weights_root()
    found = _checkpoints()
    if not found:
        return [("(no .pt files under weights)", "")]
    options = []
    for path in found:
        try:
            label = path.relative_to(root).as_posix()
        except ValueError:
            label = path.as_posix()
        options.append((label, str(path)))
    return options


def _png(rgb):
    buf = io.BytesIO()
    Image.fromarray(rgb, mode="RGB").save(buf, format="PNG")
    return buf.getvalue()


def _stamp(frame, x, y):
    """Red dot on a copy of ``frame``. The array S1 reads is unchanged."""
    out = np.array(frame, copy=True)
    height, width = out.shape[:2]
    col = int(round(float(x) * (width - 1)))
    row = int(round((1.0 - float(y)) * (height - 1)))
    y0 = max(0, row - DOT_RADIUS)
    y1 = min(height, row + DOT_RADIUS + 1)
    x0 = max(0, col - DOT_RADIUS)
    x1 = min(width, col + DOT_RADIUS + 1)
    yy, xx = np.ogrid[y0:y1, x0:x1]
    mask = (yy - row) ** 2 + (xx - col) ** 2 <= DOT_RADIUS ** 2
    out[y0:y1, x0:x1][mask] = (255, 0, 0)
    return out


def _world(x_frac, y_from_top):
    # The exported frame is y-up: the top row of the picture is world y = 1.
    x = min(1.0, max(0.0, float(x_frac)))
    y = min(1.0, max(0.0, 1.0 - float(y_from_top)))
    return x, y


def _sync():
    if device.type == "cuda":
        torch.cuda.synchronize()


_opts = _ckpt_options()
ckpt = widgets.Dropdown(
    options=_opts,
    value=_opts[0][1],
    description="Checkpoint:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="640px"),
)
refresh_btn = widgets.Button(description="Refresh")
load_btn = widgets.Button(description="Load", button_style="primary")
new_btn = widgets.Button(description="New board")
settings = settings_editor(height_px=560)
status = widgets.HTML("Load a checkpoint, then click the board.")
board = widgets.Image(format="png", width=BOARD_PX, height=BOARD_PX)
board.add_class("s1-board")
log = widgets.Output(layout=widgets.Layout(
    border="1px solid #ccc",
    height="320px",
    max_height="320px",
    overflow_y="auto",
    width="100%",
    flex="0 0 auto",
))
log.add_class("s1-log")
# The page script writes "seq,x,y" here. A hidden box still has an
# <input>, and the input event is what ipywidgets syncs to the kernel.
click_box = widgets.Text(value="")
click_box.add_class("s1-click")
click_box.layout.height = "0px"
click_box.layout.visibility = "hidden"
click_box.layout.overflow = "hidden"
controls = (ckpt, refresh_btn, load_btn, new_btn)


def _say(text):
    log.append_stdout(text if text.endswith("\n") else text + "\n")


def _set_busy(flag):
    state["busy"] = flag
    for widget in controls:
        widget.disabled = flag
    settings.set_disabled(flag)


def _show(frame, dot):
    if dot is None:
        board.value = _png(frame)
    else:
        board.value = _png(_stamp(frame, dot[0], dot[1]))


def _start_game():
    state["game"] = new_multi_gold_game(n_gold=None, opening="any")
    state["cache"] = WallCache()
    state["frame"] = state["cache"].render(state["game"])
    state["dot"] = None
    _show(state["frame"], None)


def on_refresh(_):
    selected = ckpt.value
    options = _ckpt_options()
    ckpt.options = options
    values = [value for _, value in options]
    ckpt.value = selected if selected in values else values[0]
    n = sum(1 for value in values if value)
    _say(f"{n} checkpoint(s) under weights/")


def on_load(_):
    path = ckpt.value
    if not path:
        _say("no checkpoint selected")
        return
    if state["busy"]:
        return
    _set_busy(True)
    status.value = "Loading checkpoint."
    try:
        net = S1()
        net.load_weights(path)
        net.to(device).eval()
        state["model"] = net
        n_params = sum(p.numel() for p in net.parameters())
        _say(f"loaded {path} ({n_params / 1e6:.2f}M params) on {device}")
        status.value = "Click the board."
    except Exception:
        status.value = "Load failed."
        _say(traceback.format_exc())
        raise
    finally:
        _set_busy(False)


def on_new(_):
    if state["busy"]:
        return
    _start_game()
    d = game_to_settings_dict(state["game"])
    settings.show_view(d)
    _say(
        f"new board  gold={len(d.get('gold') or [])}  "
        f"openings={len(d.get('openings') or [])}"
    )
    if state["model"] is None:
        status.value = "Load a checkpoint, then click the board."
    else:
        status.value = "Click the board."


def on_settings_edit(_):
    if state["busy"]:
        return
    settings.enter_edit(game_to_settings_dict(state["game"]))


def on_settings_render(_):
    if state["busy"] or not settings.is_editing():
        return
    try:
        state["game"] = apply_edited_settings_dict(settings.collect())
    except ValueError as exc:
        settings.set_error(f"Error, fix settings before rendering.\n{exc}")
        return
    state["cache"] = WallCache()
    state["frame"] = state["cache"].render(state["game"])
    _show(state["frame"], state["dot"])
    settings.show_view(game_to_settings_dict(state["game"]))
    _say("rendered settings")


def _pursue(x, y):
    clicks.append({"x": x, "y": y, "t": time.time()})
    state["dot"] = (x, y)
    _show(state["frame"], state["dot"])
    _say(f"click x={x:.4f} y={y:.4f}")
    model = state["model"]
    if model is None:
        _say("no checkpoint loaded")
        return

    status.value = "S1 is moving."
    game = state["game"]
    cache = state["cache"]
    frame = state["frame"]
    noop_run = 0
    i = 0
    while noop_run < NOOP_STOP and i < MAX_MOVES:
        i += 1
        _sync()
        t_think = time.perf_counter()
        image = torch.from_numpy(np.ascontiguousarray(frame))
        image = image.permute(2, 0, 1).unsqueeze(0)
        xy = torch.tensor([[x, y]], dtype=torch.float32)
        image = image.to(device, non_blocking=device.type == "cuda")
        xy = xy.to(device, non_blocking=device.type == "cuda")
        with torch.inference_mode():
            logits = model(image, xy)
        action_i = int(logits.argmax(dim=-1).item())
        logit_vals = [float(v) for v in logits[0].detach().float().cpu().tolist()]
        action = ACTIONS[action_i]
        _sync()
        think_s = time.perf_counter() - t_think

        oracle = point_oracle(game.settings, x, y)
        apply_primitive(game, action)

        t_render = time.perf_counter()
        frame = cache.render(game)
        render_s = time.perf_counter() - t_render
        state["frame"] = frame

        t_nb = time.perf_counter()
        _show(frame, (x, y))
        notebook_s = time.perf_counter() - t_nb
        moves.append({
            "i": i,
            "x": x,
            "y": y,
            "move": action,
            "oracle": oracle,
            "logits": logit_vals,
            "think_s": think_s,
            "render_s": render_s,
            "notebook_s": notebook_s,
        })
        logit_line = "  ".join(
            f"{name}={value!r}" for name, value in zip(ACTIONS, logit_vals)
        )
        _say(
            f"{i:4d}  {action:<9}  oracle {oracle:<9}  "
            f"think {think_s * 1e3:7.1f} ms  "
            f"render {render_s * 1e3:7.1f} ms  "
            f"notebook {notebook_s * 1e3:7.1f} ms\n"
            f"{logit_line}\n"
            "\n"
        )
        if action == "noop":
            noop_run += 1
        else:
            noop_run = 0

    if noop_run >= NOOP_STOP:
        _say(f"{NOOP_STOP} no-ops in a row. Click somewhere else.")
    else:
        _say(
            f"stopped after {i} moves (cap {MAX_MOVES}: half turn + "
            f"diagonal + 20%). Click somewhere else."
        )
    status.value = "Click the board."


def on_click(change):
    raw = change["new"]
    if not raw or raw.count(",") != 2:
        return
    seq_s, xs, ys = raw.split(",")
    try:
        seq = int(seq_s)
        x_frac = float(xs)
        y_top = float(ys)
    except ValueError:
        return
    if seq <= state["ignore_through"] or state["busy"]:
        return
    if settings.is_editing():
        _say("Render game settings before clicking the board.")
        return
    _set_busy(True)
    try:
        x, y = _world(x_frac, y_top)
        _pursue(x, y)
    except Exception:
        status.value = "Error. Click the board to try again."
        _say(traceback.format_exc())
        raise
    finally:
        try:
            latest = int((click_box.value or "0").split(",", 1)[0])
        except ValueError:
            latest = seq
        state["ignore_through"] = max(latest, seq)
        _set_busy(False)


refresh_btn.on_click(on_refresh)
load_btn.on_click(on_load)
new_btn.on_click(on_new)
settings.edit_btn.on_click(on_settings_edit)
settings.render_btn.on_click(on_settings_render)
click_box.observe(on_click, names="value")

_start_game()
settings.show_view(game_to_settings_dict(state["game"]))
_say(f"device {device}")
_say(
    "columns: move, oracle, think (forward), "
    "render (frame after the move), notebook (picture update); "
    "logits on the next line (noop CLOCK ANTICLOCK FORWARD)"
)
n_ckpt = sum(1 for _, value in _opts if value)
_say(f"{n_ckpt} checkpoint(s) under weights/")

# Window capture, same idea as the Shift-Enter taming in notebook_ui:
# the listener sees the click on the board <img> and writes normalized
# picture coordinates into the hidden text box, which syncs to Python.
board_col = widgets.VBox(
    [status, board],
    layout=widgets.Layout(flex="0 0 auto"),
)
display(HTML(
    """
<style>
.s1-board img { cursor: crosshair; }
.s1-log {
  height: 320px !important;
  max-height: 320px !important;
  overflow-y: auto !important;
  flex: 0 0 auto !important;
}
</style>
<script>
(function () {
  if (window.__s1BoardClick) { return; }
  window.__s1BoardClick = true;
  window.addEventListener("click", function (ev) {
    var img = ev.target;
    if (!img || img.tagName !== "IMG" || !img.closest(".s1-board")) { return; }
    var r = img.getBoundingClientRect();
    if (!(r.width > 0) || !(r.height > 0)) { return; }
    var x = (ev.clientX - r.left) / r.width;
    var y = (ev.clientY - r.top) / r.height;
    x = Math.min(1, Math.max(0, x));
    y = Math.min(1, Math.max(0, y));
    window.__s1ClickN = (window.__s1ClickN || 0) + 1;
    var box = document.querySelector(".s1-click input");
    if (!box) { return; }
    box.value = window.__s1ClickN + "," + x + "," + y;
    box.dispatchEvent(new Event("input", { bubbles: true }));
  }, true);
  if (!window.__s1LogFollow) {
    window.__s1LogFollow = true;
    setInterval(function () {
      var box = document.querySelector(".s1-log");
      if (!box) { return; }
      var gap = box.scrollHeight - box.scrollTop - box.clientHeight;
      if (gap < 80) { box.scrollTop = box.scrollHeight; }
    }, 250);
  }
})();
</script>
"""
))
display(widgets.VBox([
    widgets.HBox([ckpt, refresh_btn, load_btn, new_btn]),
    widgets.HBox([
        board_col,
        widgets.VBox(
            [settings.box],
            layout=widgets.Layout(flex="1 0 520px", min_width="520px"),
        ),
    ], layout=widgets.Layout(width="100%", align_items="flex-start")),
    log,
    click_box,
]))
